In [2]:
import librosa
import matplotlib.pyplot as plt
import librosa.display
import os
import numpy as np
from IPython.display import Audio
import wave
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import tensorflow as tf
import tensorflow_io as tfio
from IPython.display import Audio

2024-10-27 18:19:45.455023: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-10-27 18:19:45.465434: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-10-27 18:19:45.468535: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-10-27 18:19:45.477446: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-10-27 18:19:46.208844: W tensorflow/compiler/tf2

In [3]:
root_fp = "/home/alien/Git/DATA/LibriStutterData"
df = pd.read_csv("/home/alien/Git/modified_file.csv")
df.head()
df.shape

(4736, 9)

In [4]:
df['Name'] = df[df.columns[0:3]].apply(
    lambda x: '_'.join(x.dropna().astype(str)),
    axis=1
)
df.head()

,filepath,filename,NoStutteredWords,Interjection,Prolongation,Block,SoundRep,WordRep,audiofilepath,Name
0,/home/alien/Git/DATA/LibriStutterData/LibriStu...,103-1240-0051.csv,59,0,0,2,0,0,/home/alien/Git/DATA/LibriStutterData/LibriStu...,/home/alien/Git/DATA/LibriStutterData/LibriStu...
1,/home/alien/Git/DATA/LibriStutterData/LibriStu...,103-1240-0000.csv,35,0,0,0,0,1,/home/alien/Git/DATA/LibriStutterData/LibriStu...,/home/alien/Git/DATA/LibriStutterData/LibriStu...
2,/home/alien/Git/DATA/LibriStutterData/LibriStu...,103-1240-0021.csv,50,0,0,0,1,1,/home/alien/Git/DATA/LibriStutterData/LibriStu...,/home/alien/Git/DATA/LibriStutterData/LibriStu...
3,/home/alien/Git/DATA/LibriStutterData/LibriStu...,103-1240-0024.csv,49,0,0,0,1,2,/home/alien/Git/DATA/LibriStutterData/LibriStu...,/home/alien/Git/DATA/LibriStutterData/LibriStu...
4,/home/alien/Git/DATA/LibriStutterData/LibriStu...,103-1240-0049.csv,58,0,1,1,1,0,/home/alien/Git/DATA/LibriStutterData/LibriStu...,/home/alien/Git/DATA/LibriStutterData/LibriStu...


In [5]:
# "fluent"
# "soundrep"
# "wordrep"
# "block"
# "prolongation"
# "interjection"
import os
import pandas as pd
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np

# Load the CSV containing audio file paths
df = pd.read_csv('/home/alien/Git/modified_file.csv')

matplotlib.use('Agg')
# matplotlib.use('Qt5Agg')
def gen_image(fp, save_name, label):
    audio = tfio.audio.AudioIOTensor(fp)
    audio_s = audio[:]
    audio_tensor = tf.squeeze(audio_s, axis=-1)
    tensor = tf.cast(audio_tensor, tf.float32) / 32768.0
    spectrogram = tfio.audio.spectrogram(tensor, nfft=512, window=512, stride=256)
    mel_spectrogram = tfio.audio.melscale(spectrogram, rate=16000, mels=128, fmin=0, fmax=8000)
    dbscale_mel_spectrogram = tfio.audio.dbscale(mel_spectrogram, top_db=80)

    freq_mask = tfio.audio.freq_mask(dbscale_mel_spectrogram, param=10)
    time_mask = tfio.audio.time_mask(freq_mask, param=10) # FINAL

    rotated_spectrogram = np.rot90(time_mask.numpy(), k=1)

    plt.figure()
    plt.axis('off')  # Remove axes
    plt.imshow(rotated_spectrogram, cmap='magma')
    plt.tight_layout(pad=0)  # Adjust layout to remove white border

    if label == 0:
        save_name += "_fluent"
    if label == 1:
        save_name += "_stutter"
        
    print(save_name)
    plt.savefig("/home/alien/Git/DATA/LibriStutterMels/libri_mel_specaugment_soundrep/" + save_name + ".jpg", bbox_inches='tight', pad_inches=0)  # Save the MFCC plot as JPG
    # plt.show()
    plt.close()
    

# Iterate over the DataFrame rows
for index, row in df.iterrows():
    temp = row['filename'] # Assuming 'audiofilepath' is the column name
    print(temp)
    fp = row['audiofilepath']
    corresponding_sound = row['SoundRep']  # Assuming 'Interjection' is the column name
    print(corresponding_sound)

    if corresponding_sound == 0:
        gen_image(fp, temp, 0)
    elif corresponding_sound >= 1:
        gen_image(fp, temp, 1)


103-1240-0051
0


2024-10-27 18:19:47.190083: I tensorflow_io/core/kernels/cpu_check.cc:128] Your CPU supports instructions that this TensorFlow IO binary was not compiled to use: SSE3 SSE4.1 SSE4.2 AVX AVX2 FMA
I0000 00:00:1730067587.485041   11607 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1730067587.510033   11607 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1730067587.514296   11607 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at 

103-1240-0051_fluent
103-1240-0000
0
103-1240-0000_fluent
103-1240-0021
1
103-1240-0021_stutter
103-1240-0024
1
103-1240-0024_stutter
103-1240-0049
1
103-1240-0049_stutter
103-1240-0045
0
103-1240-0045_fluent
103-1240-0001
1
103-1240-0001_stutter
103-1240-0040
0
103-1240-0040_fluent
103-1240-0031
1
103-1240-0031_stutter
103-1240-0016
0
103-1240-0016_fluent
103-1240-0004
1
103-1240-0004_stutter
103-1240-0052
0
103-1240-0052_fluent
103-1240-0030
2
103-1240-0030_stutter
103-1240-0011
0
103-1240-0011_fluent
103-1240-0046
0
103-1240-0046_fluent
103-1240-0056
0
103-1240-0056_fluent
103-1240-0042
0
103-1240-0042_fluent
103-1240-0044
1
103-1240-0044_stutter
103-1240-0022
1
103-1240-0022_stutter
103-1240-0015
0
103-1240-0015_fluent
103-1240-0007
1
103-1240-0007_stutter
103-1240-0002
0
103-1240-0002_fluent
103-1240-0055
1
103-1240-0055_stutter
103-1240-0032
0
103-1240-0032_fluent
103-1240-0018
1
103-1240-0018_stutter
103-1240-0019
2
103-1240-0019_stutter
103-1240-0057
0
103-1240-0057_fluent
103-

2024-10-27 18:20:21.670172: W tensorflow/core/kernels/fft_ops.cc:552] The CUDA FFT plan cache capacity of 512 has been exceeded. This may lead to extra time being spent constantly creating new plans.


625-132118-0044_fluent
625-132118-0030
0
625-132118-0030_fluent
625-132118-0042
0
625-132118-0042_fluent
625-132118-0047
0
625-132118-0047_fluent
625-132118-0021
0
625-132118-0021_fluent
625-132118-0043
1
625-132118-0043_stutter
625-132118-0019
1
625-132118-0019_stutter
625-132118-0036
1
625-132118-0036_stutter
625-132118-0009
0
625-132118-0009_fluent
625-132118-0038
1
625-132118-0038_stutter
625-132118-0032
1
625-132118-0032_stutter
625-132118-0029
0
625-132118-0029_fluent
625-132118-0016
0
625-132118-0016_fluent
625-132118-0014
1
625-132118-0014_stutter
625-132118-0022
1
625-132118-0022_stutter
625-132118-0002
0
625-132118-0002_fluent
625-132118-0040
0
625-132118-0040_fluent
625-132118-0018
0
625-132118-0018_fluent
625-132118-0001
0
625-132118-0001_fluent
625-132118-0012
0
625-132118-0012_fluent
625-132118-0031
1
625-132118-0031_stutter
625-132118-0048
0
625-132118-0048_fluent
625-132112-0044
1
625-132112-0044_stutter
625-132112-0054
1
625-132112-0054_stutter
625-132112-0003
2
625-13